# *Pipeline* de integración continua de una *API*.

Hasta aquí se han construido *workflows* de demostración: un saludo, expresiones y la gestión de artefactos. En este cuaderno se da el paso a un *pipeline* de **integración continua** (*CI*, *Continuous Integration*): cada cambio en el repositorio instala dependencias, ejecuta un linter y pruebas automatizadas, y deja un artefacto reutilizable.

## Objetivos.

* Localizar la interfaz de programación de aplicaciones (*API*) de ejemplo en el directorio ```api/``` de este repositorio.
* Leer el *workflow* ```.github/workflows/ci.yml``` y relacionar cada paso con lo visto en los cuadernos 05, 09 y 11.
* Ejecutar el *pipeline* en un repositorio propio y comprobar que las pruebas y el artefacto aparecen en la interfaz de *GitHub Actions*.


## La aplicación de ejemplo.

El directorio ```api/``` contiene una *API* *HTTP* (*Hypertext Transfer Protocol*) escrita en *Python* con *Flask*. Expone una ruta de salud ```GET /health``` y un conjunto sencillo de rutas para gestionar tareas, en formato *JSON* (*JavaScript Object Notation*). El almacén vive en la memoria del proceso; las pruebas están en ```api/tests/``` y se ejecutan con *pytest*.

Quien cursa el taller no necesita diseñar esa aplicación: la usa como objeto del *pipeline*. Para explorarla en local, desde el directorio ```api/```:

```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements-dev.txt
cp .env.example .env
export PORT=8080
flask --app wsgi:app run --debug --port "${PORT}"
```

Las pruebas y el linter, siempre desde ```api/```:

```bash
ruff check .
pytest
```

La aplicación queda en ```http://127.0.0.1:8080```. El puerto de escucha se toma de la variable de entorno ```PORT``` (el valor de referencia está documentado en ```api/.env.example```). Las demás variables de entorno de la aplicación —incluida la clave de *Flask*— también se documentan ahí; no se versiona un archivo ```.env``` con valores reales.

En el contenedor, el proceso de arranque es *gunicorn* sobre ```wsgi:app```, con un solo *worker*.


## Qué debe hacer el *pipeline* de integración continua.

El *pipeline* de *CI* de este curso, definido en ```.github/workflows/ci.yml```, cumple estas condiciones:

* Se dispara ante un *push* o un *pull request*.
* Instala las dependencias de desarrollo (```api/requirements-dev.txt```), ejecuta *ruff* y *pytest*. Si el linter o las pruebas fallan, el *job* falla.
* Publica el informe de cobertura ```api/coverage.xml``` como artefacto ```cobertura-pytest```.
* Construye la imagen de contenedor (```docker build```) **sin** publicarla, para detectar un ```Dockerfile``` roto ya en la integración continua.

El despliegue se trata en los cuadernos 16 y 17. Aquí el objetivo es **confiar en el cambio** antes de entregarlo.


## El *workflow* ```ci.yml```.

El archivo vive en ```.github/workflows/ci.yml```. Su contenido de referencia es el siguiente.

```yaml
name: CI

on:
  push:
  pull_request:

permissions:
  contents: read

jobs:
  verificar:
    runs-on: ubuntu-latest
    defaults:
      run:
        working-directory: api
    steps:
    - name: Checkout
      uses: actions/checkout@v7

    - name: Configurar Python
      uses: actions/setup-python@v7
      with:
        python-version: "3.12"
        cache: pip
        cache-dependency-path: api/requirements-dev.txt

    - name: Instalar dependencias
      run: |
        python -m pip install --upgrade pip
        pip install -r requirements-dev.txt

    - name: Linter (ruff)
      run: ruff check .

    - name: Pruebas (pytest)
      run: pytest

    - name: Publicar informe de cobertura
      if: always()
      uses: actions/upload-artifact@v7
      with:
        name: cobertura-pytest
        path: api/coverage.xml
        if-no-files-found: ignore

    - name: Construir imagen (sin publicar)
      working-directory: .
      run: docker build -t cd441-api:ci .
```

Notas:

* ```permissions: contents: read``` aplica el principio de mínimo privilegio: este *job* no necesita escribir en el repositorio ni publicar imágenes.
* ```working-directory: api``` evita repetir ```cd api``` en los pasos de *Python*. El paso de ```docker build``` vuelve a la raíz del repositorio, porque ahí está el ```Dockerfile```.
* La versión de *Python* es la 3.12, la misma familia que usa la imagen.
* ```actions/checkout@v7```, ```actions/setup-python@v7``` y ```actions/upload-artifact@v7``` son las *majors* vigentes de esas *actions*.
* El artefacto se llama ```cobertura-pytest```. En un *job* posterior podría recuperarse con ```actions/download-artifact@v8```.


## Actividad.

1. Asegurarse de que el repositorio de trabajo contiene el directorio ```api/```, el ```Dockerfile``` en la raíz y el archivo ```.github/workflows/ci.yml```.
2. Hacer un *push* o abrir un *pull request* para disparar el *workflow* ```CI```.
3. Confirmar que el *job* ```verificar``` termina en verde y que el artefacto ```cobertura-pytest``` aparece en el resumen de la ejecución.
4. Provocar un fallo a propósito (por ejemplo, alterar una prueba de forma temporal), volver a ejecutar y comprobar que el *workflow* falla. El informe de cobertura, si llegó a generarse, puede seguir apareciendo como artefacto gracias a ```if: always()```.

Si el *workflow* no aparece, revisar que el archivo esté en ```.github/workflows/``` y que la extensión sea ```.yml``` o ```.yaml```.


## Cierre.

A partir de este cuaderno, cada cambio de la *API* queda sometido a linter, pruebas automáticas y una construcción de imagen de comprobación. El cuaderno 15 estudia el ```Dockerfile```; los cuadernos 16 y 17 publican y despliegan esa imagen.


<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2023.</p>